In [1]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
from copy import deepcopy
from copy import deepcopy

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5
from copy import deepcopy

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml as read_snap_xml

# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)
# pg.setConfigOptions(useOpenGL=False)   # 若驱动或 OpenGL 有问题可显式关闭
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

from config import DATA_DIR,INPUT_DIR


backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [2]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer


In [3]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml

# 这里，我们读取到nodes 信息,但是我们需要转化为edge信息，注意这里主要就只有包括inter-edge信息


In [4]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [start, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 30152
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)


In [5]:
from pathlib import Path


RANGES = [(0,1204),(1204,3669),(3669,4094),(4094,6814),(6814,8485),(8485,9355),(9355,10717),(10717,11640),(11640,12018),(12018,13057),
          (13057,14680),(14680,16296),(16296,18396),(18396,19831),(19831,20814),
          (20814,20822),(20822,20865),(20865,21123),(21123,22005)]

def load_nodes_range(start, end):
    path = INPUT_DIR / "modify" / f"interplane_links_{start}_{end}.xml"
    return write2xml.xml_to_nodes2(path, tegnode.tegnode)

# 一行合并所有区间
totalnode = {}
for s, e in RANGES:
    totalnode.update(load_nodes_range(s, e))

# totalnode = deepcopy(totalnode_raw)


In [6]:
start_ts =0
end_ts = 22005

In [7]:
group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [8]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 注意 ，下面是直接将nodes转为edge，因为我们的nodes本身已经完成了冲突检测和处理
import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_inter_edge = inter_edge2nodes.trans_nodes2edges(totalnode,P,N)


C:\Users\yfh\anaconda3\envs\graph_ga\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [16]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始




time_2_build = 60
# 这里，我们还需要将pending edges 信息也加入到all_inter_edge 中
pending_edges = inter_edge2nodes.trans_nodes2_pendingedges(totalnode, start_ts, end_ts, time_2_build, P, N)

In [ ]:
%%sql


In [22]:
end_ts

22005

In [10]:
# 转化为inter-edge信息后，我们可以通过绘图来初步查看

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


In [ ]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("raw behand")
viewer.resize(1200, 700)
viewer.edges_by_step = all_inter_edge

viewer.show()



注意，上述的拓扑，是只有异轨链路的，并且，在邻接表上，也是单向的。因此，我们实际上要做这几件事情

1.实际网络拓扑是还有同轨链路的，所以，我们还是要加上同轨链路信息

2.在邻接表上，我们需要将所有的边都转化为双向的

In [11]:
#添加同轨链路
# all_intra_edge = {}
# for step in range(start_ts, end_ts):
#     all_intra_edge[step] = {}
#     for i in range(P):
#         for j in range(N):
#             nownode = i * N + j
#             nextnode = i * N + (j + 1) % N
#             upnode = i * N + (j - 1 + N) % N
#             all_intra_edge[step].setdefault(nownode, set()).add(nextnode)
#             all_intra_edge[step].setdefault(nownode, set()).add(upnode)


def build_intra_edges_copies(start_ts, end_ts, P, N):
    # 预计算每个节点的左右邻居（tuple 轻量不可变，便于快速构造 set）
    base_neighbors = {
        i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
        for i in range(P) for j in range(N)
    }

    all_intra_edge = {}
    for step in range(start_ts, end_ts):
        # 一次性构造（避免 setdefault & 多次 add 的开销）
        adj = {node: set(neis) for node, neis in base_neighbors.items()}
        all_intra_edge[step] = adj
    return all_intra_edge

# 用法
all_intra_edge = build_intra_edges_copies(start_ts, end_ts, P, N)


In [12]:
def make_edges_bidirectional(edge_dict):
    """
    edge_dict: {src: set([dst, ...]), ...}
    返回双向邻接表
    """
    new_edges = {}
    for src, dsts in edge_dict.items():
        for dst in dsts:
            new_edges.setdefault(src, set()).add(dst)
            new_edges.setdefault(dst, set()).add(src)
    return new_edges
for step in all_inter_edge:
    all_inter_edge[step] = make_edges_bidirectional(all_inter_edge[step])


In [13]:
all_edges = {}

for step in range(start_ts, end_ts):
    all_edges[step] = {}
    # 先合并intra_edge
    if step in all_intra_edge:
        for src, dsts in all_intra_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)
    # 再合并inter_edge
    if step in all_inter_edge:
        for src, dsts in all_inter_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)


In [17]:

viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("prove the nodes information")
viewer.resize(1200, 700)
viewer.edges_by_step = all_edges
viewer.pending_edges = pending_edges
viewer.show()


接下来，我们可以查看在这段时间内，平均最短路径的变化情况


In [14]:
# 下面的版本快一点

import networkx as nx
import matplotlib.pyplot as plt

steps = sorted(all_edges.keys())
avg_path_lengths = []
valid_steps = []

for step in steps:
    # 构造无向图
    G = nx.Graph()
    for src, dsts in all_edges[step].items():
        for dst in dsts:
            G.add_edge(src, dst)

    # 判断group合法性
    if (
        step not in group_data
        or 0 not in group_data[step]['groups']
        or 4 not in group_data[step]['groups']
    ):
        avg_path_lengths.append(float('nan'))
        continue

    group0 = group_data[step]['groups'][0]
    group4 = group_data[step]['groups'][4]

    path_lengths = []
    # 只需要对group0的每个点跑一次BFS
    for n0 in group0:
        lengths = nx.single_source_shortest_path_length(G, n0)
        for n4 in group4:
            if n4 in lengths:
                path_lengths.append(lengths[n4])

    if path_lengths:
        avg_length = sum(path_lengths) / len(path_lengths)
        valid_steps.append(step)
    else:
        avg_length = float('nan')
    avg_path_lengths.append(avg_length)

# 可视化
plt.figure(figsize=(10, 4))
plt.plot(steps, avg_path_lengths, marker='o')
plt.xlabel('Time Step')
plt.ylabel('Group0↔Group4 平均最短路径长度')
plt.title('区域间平均最短路径随时间变化')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\yfh\AppData\Local\Temp\ipykernel_21780\1812736827.py:51: UserWarning: Glyph 8596 (\N{LEFT RIGHT ARROW}) missing from font(s) Microsoft YaHei.
  plt.tight_layout()
C:\Users\yfh\AppData\Local\Temp\ipykernel_21780\1812736827.py:52: UserWarning: Glyph 8596 (\N{LEFT RIGHT ARROW}) missing from font(s) Microsoft YaHei.
  plt.show()
